In [2]:
import joblib
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point

project_root = Path(r"C:\Users\jamad\Documents\geosense")

data_path = project_root / "data" / "processed" / "labelled_sites.csv"
model_path = project_root / "models" / "site_suitability_model.pkl"
scaler_path = project_root / "models" / "site_feature_scaler.pkl"
feature_path = project_root / "models" / "site_feature_list.pkl"

output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_gpkg = output_dir / "site_suitability_map.gpkg"
output_csv = output_dir / "site_suitability_scores.csv"

df = pd.read_csv(data_path)

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
feature_cols = joblib.load(feature_path)

X = df[feature_cols].copy()
X_scaled = scaler.transform(X)

df["good_site_probability"] = model.predict_proba(X_scaled)[:, 1]
df["opportunity_score"] = (df["good_site_probability"] * 10).round(2)
df["risk_score"] = ((1 - df["good_site_probability"]) * 10).round(2)

df["verdict"] = df["opportunity_score"].apply(
    lambda score: "GOOD SITE" if score >= 5.0 else "POOR SITE"
)

df["suitability_class"] = pd.cut(
    df["opportunity_score"],
    bins=[-0.01, 3, 5, 7, 10],
    labels=["Very Poor", "Poor", "Moderate", "Good"]
)

geometry = [
    Point(longitude, latitude)
    for longitude, latitude in zip(df["longitude"], df["latitude"])
]

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

if output_gpkg.exists():
    output_gpkg.unlink()

gdf.to_file(
    output_gpkg,
    layer="ev_site_suitability",
    driver="GPKG"
)

df.drop(columns="geometry", errors="ignore").to_csv(output_csv, index=False)

print("=" * 70)
print("SUCCESS — FINAL EV SITE SUITABILITY MAP CREATED")
print("=" * 70)
print("\nTotal candidate locations:", len(gdf))
print("\nVerdict counts:")
print(gdf["verdict"].value_counts())

print("\nSuitability class counts:")
print(gdf["suitability_class"].value_counts())

print("\nGeoPackage map file:")
print(output_gpkg)

print("\nCSV score table:")
print(output_csv)

SUCCESS — FINAL EV SITE SUITABILITY MAP CREATED

Total candidate locations: 1002

Verdict counts:
verdict
GOOD SITE    535
POOR SITE    467
Name: count, dtype: int64

Suitability class counts:
suitability_class
Good         528
Very Poor    454
Poor          13
Moderate       7
Name: count, dtype: int64

GeoPackage map file:
C:\Users\jamad\Documents\geosense\outputs\site_suitability_map.gpkg

CSV score table:
C:\Users\jamad\Documents\geosense\outputs\site_suitability_scores.csv
